<a href="https://colab.research.google.com/github/GiammaBigFishEngineer/MaskArchitectureAnomaly_CourseProject/blob/main/eomt/STEP5_lighiting_no_head.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Anomaly Detection with Mask Architecture

© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

### Clone the repository and navigate to the project directory

This section clones the project repository from GitHub and changes the current working directory to `MaskArchitectureAnomaly_CourseProject/eomt` to access the project files.

In [1]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git

%cd MaskArchitectureAnomaly_CourseProject/eomt

Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 482, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 482 (delta 35), reused 24 (delta 24), pack-reused 443 (from 3)
Receiving objects: 100% (482/482), 31.64 MiB | 16.95 MiB/s, done.
Resolving deltas: 100% (207/207), done.
/content/MaskArchitectureAnomaly_CourseProject/eomt


### Mount Google Drive

This step mounts your Google Drive to allow access to datasets, pre-trained models, and to save training checkpoints.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Configure System Path

These commands add the project's root directory and the `eomt` subdirectory to the Python system path, ensuring that necessary modules can be imported correctly.

In [3]:
import sys # Allows importing from the 'eomt' folder
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eomt')

### Install Dependencies

This cell installs all required Python packages listed in `requirements.txt`, ensuring all necessary libraries for the project are available.

In [ ]:
!python3 -m pip install -r requirements.txt

### Weights & Biases (W&B) Setup

This section configures Weights & Biases for experiment tracking. It uninstalls and upgrades `wandb` to the latest version, then logs in and initializes a logger for recording training metrics and model performance. The `log_model=False` setting prevents large model weights from being uploaded to W&B, saving bandwidth and cloud storage.

In [5]:
!pip uninstall -y wandb
!pip install -U wandb

import wandb
from lightning.pytorch.loggers import WandbLogger

wandb.login()

wandb_logger = WandbLogger(project="cityscapes_finetuning", # Crea (o seleziona, se esiste già) una cartella macro chiamata progetto sul tuo profilo online. Tutti i tentativi e i test correlati a questo lavoro verranno raggruppati qui dentro
                           name="finetune_selective_run", # Assegna un nome specifico a questa esecuzione (es. eomt_baseline_run). Ti permette di distinguere questo addestramento dai futuri test (magari con iperparametri diversi).
                           log_model = False # Impedisce a WandB di caricare i file pesanti dei pesi del modello (.ckpt o .bin) sui loro server cloud alla fine di ogni epoca. Salva direttamente sul tuo Google Drive tramite il ModelCheckpoint, risparmiando banda internet e spazio cloud gratuito su WandB.
                           )


Found existing installation: wandb 0.19.10
Uninstalling wandb-0.19.10:
  Successfully uninstalled wandb-0.19.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 63.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gianmaria-difro03 (gianmaria-difro03-politencico-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Core Setup and Utility Functions

This block imports essential libraries like `yaml`, `torch`, `lightning`, `matplotlib`, and `numpy`. It also sets a random seed for reproducibility and defines utility functions:

-   `create_mapping`: Generates a color mapping for unique class IDs.
-   `apply_colormap`: Applies the generated color map to a semantic segmentation image for visualization.

Key parameters like `device`, `IGNORE_INDEX`, `img_idx`, and `data_path` are initialized here.

## Setup

In [6]:
import yaml
import torch
import importlib
import logging
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import warnings

from lightning import seed_everything
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
from lightning.pytorch.callbacks import ModelCheckpoint

seed_everything(0, verbose=False)

device = 0  # TODO: change to the GPU you want to use
IGNORE_INDEX = 255
img_idx = 10  # TODO: change to the index of the image you want to visualize
data_path = "/content/drive/MyDrive/Cityscapes_Backup/"  # drive folder of the cityscapes val set

def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

### Load Dataset Configuration

This cell loads the dataset configuration from a YAML file, specifying the pre-trained model weights (`eomt_coco.bin`) and the model configuration (`eomt_base_640.yaml`). It then initializes the Cityscapes semantic segmentation data module, setting parameters like target image size, number of classes, batch size, and number of workers. The `data.setup()` call prepares the dataset for use.

## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [7]:
state_dict_path = "/content/drive/MyDrive/eomt_coco.bin" # punta ai pesi pre-addestrati su COCO
config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

with open(config_path, "r") as f:
    config_cs = yaml.safe_load(f)

target_img_size = (640, 640)
num_classes_final = 19  # classi di cityscapes

data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=2,
    num_workers=2,
    check_empty_targets=True,
    img_size = target_img_size,
    **data_module_kwargs
)
data.setup()

### Model Setup and Selective Freezing

This section configures the neural network model. It initializes the encoder and the EoMT (Mask Architecture Anomaly) network with specific parameters, including `num_q=200` to match COCO pre-trained weights and `num_classes=19` for Cityscapes. It then initializes the Lightning Module, loads the pre-trained weights, and performs selective freezing of model layers. Only the `class_head` is unfrozen, allowing it to learn the 19 Cityscapes classes while keeping other parts of the model fixed during fine-tuning.

In [8]:
# 1. Setup Network (FORZA query a 200 per compatibilità COCO)
net_cfg = config_cs["model"]["init_args"]["network"]
encoder_cfg = net_cfg["init_args"]["encoder"]

# Inizializza Encoder
enc_mod, enc_cls = encoder_cfg["class_path"].rsplit(".", 1)
encoder = getattr(importlib.import_module(enc_mod), enc_cls)(img_size=(640,640), **encoder_cfg.get("init_args", {}))

# Inizializza Network (EoMT)
net_mod, net_cls = net_cfg["class_path"].rsplit(".", 1)
network = getattr(importlib.import_module(net_mod), net_cls)(
    masked_attn_enabled=True, #MODIFICATO
    num_classes=19, # Cityscapes
    encoder=encoder,
    num_q=200,      # Fondamentale: deve matchare i pesi COCO
    num_blocks=3
)

# 2. Setup Lightning Module
lit_mod, lit_cls = config_cs["model"]["class_path"].rsplit(".", 1)

model_kwargs_final = {k: v for k, v in config_cs["model"]["init_args"].items() if k != "network"}
model_kwargs_final.pop("num_classes", None)
model_kwargs_final.pop("img_size", None)

model = getattr(importlib.import_module(lit_mod), lit_cls)(
          img_size=target_img_size,
          num_classes=num_classes_final,
          network=network,
          **model_kwargs_final,
          ckpt_path=state_dict_path,          # Serve alla TUA classe per caricare il .bin iniziale
          load_ckpt_class_head=False
      ).to(device)

#print(model)




# 3. FREEZING SELETTIVO
for param in model.parameters():
    param.requires_grad = False

# Sblocca Testa + Ultimo Blocco Decoder
layers_to_train = ["class_head", #sblocco la testa --> Impara i nomi delle 19 classi di Cityscapes
                   #"mask_head", #--> Impara le geometrie e i contorni degli oggetti stradali
                   #"network.upscale" #--> Adatta la risoluzione spaziale alle immagini urbane
                   #"network.q" # sblocca le query
                   ]
for name, param in model.named_parameters():
    if any(k in name for k in layers_to_train):
        param.requires_grad = True
        print(f"SBLOCCATO: {name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


SBLOCCATO: network.class_head.weight
SBLOCCATO: network.class_head.bias


### Training Configuration and Checkpointing

This block sets up the training process using `lightning.pytorch.callbacks.ModelCheckpoint` to save the best model based on validation IoU. It defines the path and filename for checkpoints and specifies that only the top-performing model should be saved. A `L.Trainer` instance is configured with parameters like `max_epochs`, `accelerator`, `precision`, `logger` (WandB), and `callbacks`. It also includes logic for resuming training from a checkpoint if `use_checkpoint` is set to `True`.

In [9]:
from lightning.pytorch.callbacks import ModelCheckpoint

# 1. Definiamo il Checkpoint Callback
checkpoint_callback = ModelCheckpoint(
    dirpath="/content/drive/MyDrive/Modelli_EoMT", # Dove salvare
    filename="eomt-cityscapes-finetuning-head_complete-{epoch:02d}-{metrics/val_iou_all:.3f}",
    monitor="metrics/val_iou_all", # La metrica da monitorare
    mode="max",                    # Vogliamo massimizzare la IoU
    save_top_k=1,                  # Salva solo il file migliore in assoluto
    save_last = True,
    verbose=True                   # ??
)

current_max_epochs = 10
use_checkpoint = True ##########################################
resume_ckpt_path = "/content/drive/MyDrive/Modelli_EoMT/last.ckpt"


# 2. Configuriamo il Trainer includendo il callback e il logger
trainer = L.Trainer(
    max_epochs=current_max_epochs,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger,          # Fondamentale per vedere i grafici
    callbacks=[checkpoint_callback], # Colleghiamo il salvataggio automatico

    #limit_train_batches=0.01,      # Opzionale: per test rapidi
    #limit_val_batches=0.01,        # Valida solo sul 5%

    log_every_n_steps=10,
    default_root_dir="/content/drive/MyDrive/Modelli_EoMT/"
)

# 3. Avvio Training (GESTIONE DEL CHECKPOINT)
if use_checkpoint:
  print(f"Ripresa dell'addestramento dal checkpoint: {resume_ckpt_path}")
  trainer.fit(model, datamodule=data, ckpt_path=resume_ckpt_path)
else:
  trainer.fit(model, datamodule=data)

# 4. Dopo il training, stampa dove si trova il file migliore
print(f"Miglior modello salvato in: {checkpoint_callback.best_model_path}")
print(f"Miglior mIoU raggiunta: {checkpoint_callback.best_model_score:.4f}")



INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


Ripresa dell'addestramento dal checkpoint: /content/drive/MyDrive/Modelli_EoMT/last.ckpt


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/Modelli_EoMT exists and is not empty.
INFO: Restoring states from the checkpoint path at /content/drive/MyDrive/Modelli_EoMT/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/MyDrive/Modelli_EoMT/last.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
INFO:lightning.pytorch.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
INFO: 
  | Name      | Type                   | Params | Mode 
-------------------------------------------------------------
0 | network   | EoMT                   | 93.6 M | train
1 | criterion | MaskClassificationLoss | 0      | train
2 | metrics   | ModuleList             | 0 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


Miglior modello salvato in: /content/drive/MyDrive/Modelli_EoMT/eomt-cityscapes-finetuning-head_complete-epoch=08-metrics/val_iou_all=0.635.ckpt
Miglior mIoU raggiunta: 0.6348


### Convert Lightning Checkpoint to `.bin` Format

This cell converts the PyTorch Lightning checkpoint (`.ckpt`) into a standard `.bin` file, which contains only the model's `state_dict`. This is useful for deploying the model or using it with other frameworks that expect a simpler weight format. It bypasses PyTorch's `weights_only` security block, extracts the `state_dict`, and cleans up any `model.` prefixes in the keys for broader compatibility.

In [11]:
import os
import torch
from functools import partial

# 1. Bypassiamo il blocco di sicurezza di PyTorch 2.6+
torch.load = partial(torch.load, weights_only=False)

# 2. Definiamo i percorsi dei file
checkpoint_path = "/content/drive/MyDrive/Modelli_EoMT/last.ckpt"  # Il tuo file sorgente
output_bin_path = "/content/drive/MyDrive/Modelli_EoMT/eomt_model_weights.bin"  # Il file .bin finale

print(f"Caricamento del checkpoint Lightning da: {checkpoint_path}...")

try:
    # 3. Carichiamo il checkpoint completo (contiene epoch, optimizers, state_dict, ecc.)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

    # Estrarre lo state_dict puro
    lightning_state_dict = checkpoint["state_dict"]

    # 4. Pulizia del prefisso 'model.' (opzionale ma caldamente consigliata per compatibilità)
    # Se il tuo LightningModule conteneva l'architettura dentro 'self.model', i pesi avranno chiavi tipo 'model.encoder...'
    clean_state_dict = {}
    for key, value in lightning_state_dict.items():
        if key.startswith("model."):
            new_key = key.replace("model.", "", 1) # Rimuove solo il primo prefisso 'model.'
            clean_state_dict[new_key] = value
        else:
            clean_state_dict[key] = value

    # 5. Salvataggio in formato standard .bin
    print(f"Salvataggio dello state_dict in formato .bin...")
    torch.save(clean_state_dict, output_bin_path)

    print("-" * 50)
    print(f"Conversione completata con successo!")
    print(f"File generato: {output_bin_path}")
    print(f"Numero di tensori salvati: {len(clean_state_dict)}")
    print("-" * 50)

except FileNotFoundError:
    print(f"Errore: Non ho trovato nessun file in '{checkpoint_path}'. Controlla il percorso.")
except Exception as e:
    print(f"Si è verificato un errore durante la conversione: {str(e)}")

Caricamento del checkpoint Lightning da: /content/drive/MyDrive/Modelli_EoMT/last.ckpt...
Salvataggio dello state_dict in formato .bin...
--------------------------------------------------
Conversione completata con successo!
File generato: /content/drive/MyDrive/Modelli_EoMT/eomt_model_weights.bin
Numero di tensori salvati: 198
--------------------------------------------------


### Install GCN Library

This cell installs the `gcn` library, which might be a dependency for certain graph-based operations or utilities used in the evaluation phase of the model.

# Evaluation

In [12]:
!pip install gcn

### Inference and Plotting Utility Functions

This section defines two key functions:

-   `infer_semantic`: Performs semantic segmentation inference on an image using sliding windows. It handles image preprocessing, forward pass through the model, and recomposition of full logits from windowed predictions. It returns the predicted segmentation mask and the ground truth mask.
-   `plot_semantic_results`: Visualizes the original image, the model's semantic prediction, and the ground truth segmentation using a custom colormap. It ensures images are correctly normalized for display.

In [13]:
def infer_semantic(img, target, model, target_size=(640, 640)):
    """
    Esegue l'inferenza semantica con sliding window e pulisce le dimensioni extra.
    """
    model.eval()

    # Se l'immagine ha una forma tipo [1, 3, 640, 640] presa dalla lista,
    # rimuoviamo il batch iniziale per dare al modello il formato [3, 640, 640] corretto
    if len(img.shape) == 4 and img.shape[0] == 1:
        img = img.squeeze(0)

    img = img.to(device)

    # Impostiamo la dimensione della finestra del modello
    model.window_size = target_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img]
        img_sizes = [img.shape[-2:]]

        # Divisione in finestre (sliding window)
        crops, origins = model.window_imgs_semantic(imgs)

        # Forward pass
        mask_logits_layers, class_logits_layers = model(crops)

        # Prendiamo l'output dell'ultimo blocco del decoder
        m_logits = F.interpolate(mask_logits_layers[-1], target_size, mode="bilinear")
        c_logits = class_logits_layers[-1]

        # Conversione query-mask -> pixel-logits
        crop_pixel_logits = model.to_per_pixel_logits_semantic(m_logits, c_logits)

        # Ricomposizione dell'immagine originale dalle finestre
        full_logits = model.revert_window_logits_semantic(crop_pixel_logits, origins, img_sizes)

        # Selezione della classe dominante per ogni pixel
        preds = full_logits[0].argmax(0).cpu().numpy()

    # Conversione del Ground Truth per il confronto
    target_pixel = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()

    # ASSICURIAMOCI CHE SIANO INTERI E SENZA DIMENSIONI "1" ORFANE
    return preds.squeeze().astype(np.int32), target_pixel.squeeze().astype(np.int32)




def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Gestione dell'immagine (se normalizzata o meno)
    img_np = img.permute(1, 2, 0).cpu().numpy()
    # Se i valori sono fuori dal range [0,1], normalizziamo per la visualizzazione
    if img_np.max() > 1.0 or img_np.min() < 0:
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

    axes[0].imshow(img_np) # Ora usiamo l'immagine processata
    axes[0].set_title("Original Image")

    # 2. Applica la colormap alla predizione
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Model Prediction")

    # 3. Applica la colormap al target (Ground Truth)
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Ground Truth")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

### Model Evaluation and IoU Calculation

This block evaluates the fine-tuned model on the validation dataset. It iterates through the validation loader, performs semantic inference using the `infer_semantic` function, and computes a confusion matrix using `fast_hist`. Finally, it calculates the Intersection over Union (IoU) for each Cityscapes class and the mean IoU (mIoU) across all classes. The results are then printed in a formatted table, comparing the performance of the fine-tuned model.

In [14]:
import numpy as np
import torch
import gcn
from tqdm import tqdm


# Funzioni di calcolo (rimangono invariate nella logica)
def fast_hist(a, b, n):
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n)
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    return np.diag(hist) / (np.maximum(1.0, hist.sum(1) + hist.sum(0) - np.diag(hist)))


# =====================================================================
# 1. SETUP DELLE MATRICI DI CONFUSIONE E DEI MODELLI
# =====================================================================
num_eval_classes = 19
val_loader = data.val_dataloader()

hist_finetuned = np.zeros((num_eval_classes, num_eval_classes))

# --- STEP A: VALUTAZIONE CON IL MODELLO FINE-TUNED ---
print(f"Fase 1: Valutazione in corso con il modello FINE-TUNED su {len(val_loader.dataset)} immagini...")

# FONDAMENTALE: Assicuriamoci che tutto il modello sia sulla GPU corretta
model = model.to(device)
model.eval()

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Fine-tuned Evaluation"):
        imgs, targets = batch

        for j in range(len(imgs)):
            # Inferenza con il modello Fine-Tuned
            pred_ft, target_ft = infer_semantic(
                imgs[j],
                targets[j],
                model,
                target_size=target_img_size
            )
            hist_finetuned += fast_hist(target_ft.flatten(), pred_ft.flatten(), num_eval_classes)

# =====================================================================
# 2. CALCOLO DEI RISULTATI FINALI (IoU e mIoU)
# =====================================================================
ious_finetuned = per_class_iu(hist_finetuned)
miou_finetuned = np.nanmean(ious_finetuned)

# Classi ufficiali Cityscapes
classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

# =====================================================================
# 3. VISUALIZZAZIONE TABELLA COMPARATIVA CONSOLIDATA
# =====================================================================
print("\n" + "="*30)
print(f"{'CLASSE CITYSCAPES':<22} | {'IoU FINE-TUNED %':<15}")
print("-" * 30)

for name, iou_ft in zip(classes, ious_finetuned):
    ft_p = iou_ft * 100
    print(f"{name:<22} | {ft_p:>14.2f}%")

print("-" * 30)
print(f"{'MEAN IoU GLOBAL':<22} | {miou_finetuned*100:>14.2f}%")
print("="*30)

📊 Fase 1: Valutazione in corso con il modello FINE-TUNED su 500 immagini...


Fine-tuned Evaluation: 100%|██████████| 250/250 [03:12<00:00,  1.30it/s]


CLASSE CITYSCAPES      | IoU FINE-TUNED %
------------------------------
road                   |          96.41%
sidewalk               |          73.61%
building               |          83.38%
wall                   |          55.63%
fence                  |          48.60%
pole                   |          10.46%
traffic light          |          56.61%
traffic sign           |          37.31%
vegetation             |          85.87%
terrain                |          54.47%
sky                    |          89.96%
person                 |          65.28%
rider                  |          18.98%
car                    |          89.38%
truck                  |          64.70%
bus                    |          79.94%
train                  |          65.22%
motorcycle             |          58.37%
bicycle                |          68.78%
------------------------------
MEAN IoU GLOBAL        |          63.31%
